# How to work with COSMO data

---

COSMO data are available between 2017 and 2023.

The forecasts are computed once every 3 hours and have a lead time of 34 hours. 

In order to get hourly data, we always take COSMO forecast for lead times 0,1 and 2. 

This gives us the most precise hourly forecast data we can get from the model.

Each file contains forecast for 1 day in 1-hourly resolution.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

### Prepare utility functions to load pre-processed COSMO data

COSMO has been a single model until 17.9.2020, and an ensemble of 11 forecasts since 18.9.2020.

Therefore, the <u>files until 17.9.2020</u> have the following shape:

(224, 320, 24, 8) -> (grid_height, grid_width, hours, num_variables)

and <u>files from 18.9.2020</u> have the following shape:

(224, 320, 24, 8, 2) -> (grid_height, grid_width, hours, num_variables, ensemble_mean_and_stddev), where ensemble_mean is at location [0] and emsemble_stddev at dimension [1]

In [ ]:
import h5py

# The dictionary of features available in each file
feature_name_to_index = {
    'tp': 0,            # Total Precipitaton        [mm]
    'ws': 1,            # Wind Speed                [m/s]
    'wdir': 2,          # Wind Direction            [°]
    '2t': 3,            # 2m Temperature            [°C]
    '2d': 4,            # 2m Dewpoint Temperature   [°C]
    'nswrs': 5,         # Net Shorware Radiation    [W/m^2]
    'nlwrs': 6,         # Net Longwave Radiation    [W/m^2]
    'vis': 7            # Visibility                (not sure what exactly this is for, I am not using it)
}

# Function to load the forecasts array for 1 day (24 hours) from file
# Already takes care of handling whether the COSMO forecast was a single prediction or an ensemble
# In case of an ensemble, only mean is returned
# The array is additionally reshaped so that hours are at dimension 0
# This means that this function always returns an array of shape (24, 224, 320, 8)
def load_cosmo_grid(data_path : str, year : int, month : int, day : int) -> np.array:
    # Load data from file
    hf = h5py.File(data_path + '%04i/%04i%02i%02i.h5' % (year, year, month, day), 'r')
    hf.keys()
    cosmo_grid = np.array(hf.get('cosmo_grid'))

    # Take only mean prediction
    if len(cosmo_grid.shape) == 5:
        cosmo_grid = cosmo_grid[..., 0].transpose(2, 0, 1, 3)
    else:
        cosmo_grid = cosmo_grid.transpose(2, 0, 1, 3)
    cosmo_grid[:, :, :, [3, 4]] -= 273.15  # Convert temperatures from Kelvins into Degrees celsius
    
    return cosmo_grid 

### Load data examples and print their shapes

Note that independently of whether COSMO or COSMO-ENS is used, the function load_cosmo_grid returns the same arrays (data internally pre-processed).

In [ ]:
data_path = '/storage/lukovic/Data/FORWARDS/treenet/COSMO_FromCirrus/'

In [ ]:
day_grid = load_cosmo_grid(data_path, 2020, 9, 17)
print(day_grid.shape)                                   # Before ensemble COSMO, dimensions reshuffled to have hours first
day_grid = load_cosmo_grid(data_path, 2020, 9, 18)  
print(day_grid.shape)                                   # After ensemble COSMO, dimensions reshuffled to have hours first, only mean extracted

### COSMO model corrensponding DHM (Digital Height Model)

We extract COSMO forecasts on the grid which corresponds to a DHM which we have available.

This DHM has resolution of 224 (height) x 320 (width) pixels.

The DHM is provided as a separate file with shape (height, width, 3), where:
height_map[..., 0] - latitudes
height_map[..., 1] - longitudes
height_map[..., 2] - elevations

In [ ]:
import pickle

with open(data_path + 'height_map.pkl', 'rb') as f:
    height_map = pickle.load(f)

In [ ]:
print(height_map.shape)

print(height_map[..., 0])
print(height_map[..., 1])
print(height_map[..., 2])

### Plot an example of the COSMO data

In [ ]:
%pip install plotly

In [ ]:
import plotly.express as px

# Pick 0-th hour of the day (midnight)
# Flip height dimension (otherwise Swiss Grid would be plotted upside-down)
# Pick 3-rd variable in the array (index 3 is 2m temperature as per dictionary in one of the cells above)
px.imshow(day_grid[0, ::-1, :, 3])

In [ ]:
# Plot the corresponding DHM elevation image
# Don't forget to flip height dimension (otherwise Swiss Grid would be plotted upside-down)
px.imshow(height_map[::-1, :, 2])

### Plot the DHM

Here we use Scattergeo function which allows us to plot data for particular GPS locations, and not necessarily as an image. 

This way we can demonstrate that the height_map array really stores GPS coordinates with corresponding terrain elevations.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scattergeo(
        lon = height_map[..., 1].flatten(),
        lat = height_map[..., 0].flatten(),
        mode = 'markers',
        marker_color = height_map[..., 2].flatten(),
    ),
)
fig.update_layout(
        title = 'Digital height Model',
        geo_scope='europe',
        geo_showland=False,
        geo_projection_scale=20,
        geo_center=dict(
            lon=height_map[..., 1].flatten().mean(),
            lat=height_map[..., 0].flatten().mean()
        )
    )

fig.show()

### Locating closest point on the grid w.r.t. given GPS location

First we compute the distance map...

In [ ]:
location = np.array([47.362950, 8.454470]) # Coordinates from metadata of a tree in Birmensdorf

height, width, chans = height_map.shape
hmap = height_map.reshape(-1, chans)[:, :2]
all_dists = np.sqrt(((hmap[None, :, :] - location[None, None, :])**2).sum(axis=2)) # (1, num_pts)
        
print(all_dists)

Plot the distance map to visualize distance of each point in the Digital Height Model w.r.t. our target point (Birmensdorf)

In [ ]:
px.imshow(all_dists.reshape(height, width).astype(np.float32)[::-1, :])

Get location of the target point (smallest distance) in the height_map array and use it to extract corresponding COSMO forecasts.

In [ ]:
# Extract the index of location with minimum distance
min_idx = all_dists.argmin()

# Get data from COSMO
cosmo_grid = load_cosmo_grid(data_path, 2020, 9, 18)  
print(cosmo_grid.shape)                                  
# Reshape COSMO forecasts to flatten across spatial dimensions
cosmo_flat = cosmo_grid.reshape(cosmo_grid.shape[0], -1, cosmo_grid.shape[3])
print(cosmo_flat.shape)

# Use the min-distance index to extract COSMO variables for the desired location
cosmo_location = cosmo_flat[:, min_idx, :]
print(cosmo_location.shape)

### Plot time-series example for the 24 hours at the given location, for e.g. temperature (index 3)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=np.arange(cosmo_location.shape[0]),
        y=cosmo_location[:, 3]
    )
)
fig.update_layout(
    xaxis=dict(
        title='Time [hour]'
    ),
    yaxis=dict(
        title='2m Temperature [°C]'
    )
)
fig.show()

In [1]:
import pickle
import pandas as pd
clima_l2 = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_meteo_l2_dictionary.pkl")
clima_lm = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_meteo_lm_dictionary.pkl")
temperature_l0 = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_all_l0_temperature_dictionary.pkl")
temperature_l1 = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_all_l1_temperature_dictionary.pkl")
humidity_l0 = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_all_l0_humidity_dictionary.pkl")
humidity_l1 = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_all_l1_humidity_dictionary.pkl")

In [2]:
meta = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_all.pkl")
meta_temperature_l0 = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_data_all_l0_temperature.pkl")
meta_temperature_l1 = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_data_all_l1_temperature.pkl")
meta_humidity_l0 = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_data_all_l0_humidity.pkl")
meta_humidity_l1 = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/metadata_data_all_l1_humidity.pkl")
meta

,measure_point,series_id,series_start,series_stop,series_cutout,series_active,series_display,variable_id,variable_name,variable_resolution,...,site_annual_rad,site_growth_rad,site_annual_relh,site_growth_relh,site_annual_vpd,site_growth_vpd,site_n_depo,site_ozon,site_nfk,site_temp_ref
0,alvaneu-high_1_tree-stem-radius-change_1.5_alv...,1,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,166.52,243.83,70.81,70.57,0.30,0.48,None,None,106.00,1178.0
1,alvaneu-high_2_tree-stem-radius-change_1.5_alv...,2,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,166.52,243.83,70.81,70.57,0.30,0.48,None,None,106.00,1178.0
2,alvaneu-low_3_tree-stem-radius-change_1.5_alva...,3,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,164.93,239.27,70.23,69.62,0.33,0.51,None,None,191.00,1178.0
3,alvaneu-low_4_tree-stem-radius-change_1.5_alva...,4,2014-05-21,2022-10-05,None,no,no,3,tree stem radius change,10,...,164.93,239.27,70.23,69.62,0.33,0.51,None,None,191.00,1178.0
4,bachtel-forest_5_precipitation_2_bachtel-0.pre...,5,2012-08-24,2022-09-28,None,no,no,10,precipitation,10,...,137.28,223.03,77.13,74.20,0.30,0.50,None,None,None,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1905,bern-drakau_1929_soil-water-potential_-0.4_227...,1929,2025-03-25,NaT,None,yes,yes,32,soil water potential,10,...,None,None,None,None,None,None,None,None,None,NaN
1906,bern-drakau_1930_soil-water-potential_-1_22747...,1930,2025-03-25,NaT,None,yes,yes,32,soil water potential,10,...,None,None,None,None,None,None,None,None,None,NaN
1907,bern-drakau_1931_soil-water-potential_-0.1_227...,1931,2025-03-25,NaT,None,yes,yes,32,soil water potential,10,...,None,None,None,None,None,None,None,None,None,NaN
1908,bern-drakau_1932_soil-water-potential_-0.4_227...,1932,2025-03-25,NaT,None,yes,yes,32,soil water potential,10,...,None,None,None,None,None,None,None,None,None,NaN


In [ ]:
for key, data in temperature_l0.items():
    print(key)
    climate_data = clima[meta_id[key].site_id]
    if len(data) > 0 and len(climate_data) > 0:
        a = pd.merge(data, climate_data, on = "ts", how = "outer")  # NOTE: merge the dendrometer data with the climate data
        b = a [a.value.first_valid_index():a.value.last_valid_index()]  # NOTE: remove the head and tail where "value" (in this case, the dendrometer value) is not defined
        if b.temp.isnull().all() and b.vpd.isnull().all():  # NOTE: make sure that the merged datasets actually overlap, at least over temperature and vapour pressure difference
            c = b[["series_id", "site_id", "ts", "value", "temp", "rh", "swp", "total_precip", "rad", "vpd"]].rename(columns={"value": "stem_radius"})  # NOTE: select the features to use
            c['site_id'] = c['site_id'].ffill().bfill()  # NOTE: the merging process changes the site and seris ids to double because of the presence of NaNs. This command fills the rows with the corresponding index.
            c['series_id'] = c['series_id'].ffill().bfill()
            
            c['site_id'] = c['site_id'].astype(int) # NOTE: makes sure that the values are integers
            c['series_id'] = c['series_id'].astype(int)

            start = c.ts.iloc[0]
            end = c.ts.iloc[-1]
            complete_times = {'ts': pd.date_range(start=start, end=end, freq='1h')}   # NOTE: alternative: "10Min". For more information see https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#timeseries-offset-aliases
            d = pd.merge(c, pd.DataFrame(complete_times), on = "ts", how = "outer")  # NOTE: make sure that all the time stamps are present
            df[key] = d

{7:                         ts  series_id             value
 0      2022-09-28 17:30:27          7  40.6485084306096
 1      2022-09-28 17:40:55          7   40.674448767834
 2      2022-09-28 18:00:45          7  40.6118867780575
 3      2022-09-28 18:10:47          7  40.6378271152819
 4      2022-09-28 18:30:47          7  40.5752651255055
 ...                    ...        ...               ...
 133501 2025-04-21 23:18:52          7   98.304722667277
 133502 2025-04-21 23:28:51          7  98.4847791256581
 133503 2025-04-21 23:38:52          7  98.5214007782101
 133504 2025-04-21 23:48:55          7  98.5992217898833
 133505 2025-04-21 23:58:56          7  98.5793850614176
 
 [133506 rows x 3 columns],
 8:                         ts  series_id             value
 0      2012-08-24 07:50:13          8           74.1762
 1      2012-08-24 07:56:39          8           74.4523
 2      2012-08-24 08:03:05          8           73.8947
 3      2012-08-24 08:09:45          8           76.

In [ ]:
clima_lm = pd.read_pickle("/storage/lukovic/Data/FORWARDS/treenet/server_data/data_meteo_lm_dictionary.pkl")

In [ ]:
clima_lm[1]

In [ ]:
data_all_l1[1]

In [1]:
import psycopg2
import pickle

cnx = psycopg2.connect(user = 'tnreader',
                       password = 'Iku8Gem3',
                       host = 'pgdboapp.wsl.ch',
                       port = 5432,
                       database = 'treenet')

print('connection made...')

cursor_cnx = cnx.cursor()

# query = "SELECT ts,value FROM data_dendro_lm WHERE series_id=1 LIMIT 10" 

query = "SELECT * FROM data_all_l0"

with cursor_cnx as cursor:
    cursor.execute(query)
    output = []
    #colnames = [desc[0] for desc in cursor.description]
    result = cursor.fetchall()
    for row in result:
        output.append(row)

with open('/home/lukovic/data/treenet/server_data/data_all_l0.pkl', 'wb') as f: 
    pickle.dump(output, f)

cursor_cnx.close()
cnx.close()

connection made...


: 

In [1]:
import pickle
import psycopg2
# https://www.postgresqltutorial.com/postgresql-tutorial/postgresql-where/

cnx = psycopg2.connect(user = 'tnreader',
                       password = 'Iku8Gem3',
                       host = 'pgdboapp.wsl.ch',
                       port = 5432,
                       database = 'treenet')

print('connection made...')

cursor_cnx = cnx.cursor()

query = "SELECT * FROM data_all_l1 LIMIT 100" 
#query = "SELECT * FROM INFORMATION_SCHEMA.COLUMNS --WHERE TABLE_NAME = data_dendro_lm LIMIT 10"

with cursor_cnx as cursor:
    cursor.execute(query)
    colnames = [desc[0] for desc in cursor.description]
    result = cursor.fetchall()
    
# with open('/Users/lukovic/Data/FORWARDS/TNT/raw_data/metadata_Server.pkl', 'wb') as f:
#    pickle.dump(output, f)


cursor_cnx.close()
cnx.close()


connection made...


In [ ]:
result

In [3]:
data_all_l0 = pd.read_pickle("/home/lukovic/data/treenet/server_data/data_all_l0temp.pkl")
data_all_l1 = pd.read_pickle("/home/lukovic/data/treenet/server_data/data_all_l1temp.pkl")


FileNotFoundError: [Errno 2] No such file or directory: '/home/lukovic/data/treenet/server_data/data_all_l0temp.pkl'